# 한국 소설 RAG - 데이터 수집 (알라딘 API)

알라딘 Open API로 한국 현대소설 메타데이터를 수집합니다.

**수집 대상**: 제목, 저자, 출판사, 출판일, 줄거리, 표지 이미지, ISBN


In [ ]:
!pip install requests pandas tqdm python-dotenv -q

In [ ]:
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
import requests
import pandas as pd
from tqdm import tqdm

load_dotenv(override=True)
print("✅ 라이브러리 로드 완료")


## 1. 설정

In [ ]:
# .env에 추가
# ALADIN_API_KEY=your-aladin-ttbkey

ALADIN_KEY = os.getenv("ALADIN_API_KEY")
BASE_URL   = "https://www.aladin.co.kr/ttb/api/ItemSearch.aspx"
OUTPUT_DIR = Path("../data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'✅' if ALADIN_KEY else '❌ None'} ALADIN_API_KEY")


## 2. 수집 함수

In [ ]:
def fetch_novels(query: str, page: int = 1, max_results: int = 50) -> dict:
    """
    알라딘 API 호출
    - QueryType: Keyword (키워드 검색)
    - SearchTarget: Book
    - CategoryId: 1 (국내도서) → 소설/시/희곡 카테고리
    """
    params = {
        "ttbkey":      ALADIN_KEY,
        "Query":       query,
        "QueryType":   "Keyword",
        "MaxResults":  max_results,
        "start":       page,
        "SearchTarget":"Book",
        "CategoryId":  50921,   # 한국소설 카테고리 ID
        "output":      "js",    # JSON
        "Version":     "20131101",
        "Cover":       "Big",   # 표지 이미지 크기
    }
    try:
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        # 알라딘 응답은 BOM 포함 UTF-8
        resp.encoding = "utf-8"
        return resp.json()
    except Exception as e:
        print(f"[ERROR] 쿼리: {query}, 페이지: {page} → {e}")
        return {}


def parse_novels(data: dict) -> list[dict]:
    """알라딘 응답에서 필드 추출"""
    items = data.get("item", [])
    novels = []
    for item in items:
        title = item.get("title", "").strip()
        if not title:
            continue
        # 웹소설/라이트노벨 제외
        category = item.get("categoryName", "")
        if any(kw in category for kw in ["웹소설", "라이트노벨", "만화"]):
            continue

        novels.append({
            "id":          item.get("isbn13", item.get("isbn", "")).strip(),
            "title":       title,
            "author":      item.get("author", "").strip(),
            "publisher":   item.get("publisher", "").strip(),
            "pub_year":    item.get("pubDate", "")[:4],
            "kdc_name":    category,
            "summary":     item.get("description", "").strip(),
            "keywords":    "",   # 알라딘은 키워드 미제공 → 임베딩 시 summary만 사용
            "isbn":        item.get("isbn13", "").strip(),
            "img_url":     item.get("cover", "").strip(),
        })
    return novels


## 3. 응답 구조 확인 (필수)

In [ ]:
# 실제 응답 구조 확인
sample = fetch_novels("현대소설", page=1, max_results=3)
print("최상위 키:", list(sample.keys()))
print("\n전체 응답:")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:3000])


## 4. 전체 수집

In [ ]:
# 검색 쿼리 목록 (카테고리 ID 50921 + 키워드 조합)
QUERIES = [
    "한국소설", "현대소설", "장편소설", "단편소설",
    "가족소설", "성장소설", "역사소설", "사회소설",
]

all_novels = []
seen_ids = set()

for query in QUERIES:
    print(f"\n📚 쿼리: [{query}]")
    for page in range(1, 4):  # 페이지당 50건 × 3페이지 = 최대 150건/쿼리
        data = fetch_novels(query, page=page, max_results=50)
        novels = parse_novels(data)

        if not novels:
            print(f"  {page}페이지 결과 없음 → 종료")
            break

        new_count = 0
        for n in novels:
            key = n["isbn"] or n["title"]
            if key and key not in seen_ids:
                seen_ids.add(key)
                if not n["summary"]:
                    n["summary"] = n["title"]
                all_novels.append(n)
                new_count += 1

        total = data.get("totalResults", "?")
        print(f"  페이지 {page} | 신규 {new_count}건 | 누적 {len(all_novels)}건 (전체 {total}건)")
        time.sleep(0.3)

print(f"\n✅ 수집 완료: {len(all_novels)}건 (중복 제거 후)")


## 5. 저장 및 확인

In [ ]:
# JSON 저장
output_path = OUTPUT_DIR / "novels_raw.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_novels, f, ensure_ascii=False, indent=2)
print(f"✅ JSON 저장: {output_path}")

# DataFrame 확인
df = pd.DataFrame(all_novels)
print(f"\n📊 shape: {df.shape}")
print("\n=== 출판년도 분포 ===")
print(df["pub_year"].value_counts().head(10))
print("\n=== 결측값 ===")
print(df[["title","author","summary","isbn"]].isnull().sum())
print("\n=== 샘플 ===")
df[["title","author","pub_year","summary"]].head(5)
